# Relatedness

In this exercise you will run a complete relatedness estimation pipeline on a real SNP dataset. By the end you should be able to:

- Prepare genomic data for relatedness analysis (LD pruning, filtering)
- Estimate pairwise kinship with KING-robust (PLINK2)
- Compute a 2D site frequency spectrum with ANGSD using genotype likelihoods
- Run relateAdmix for admixture-aware relatedness
- Visualise and compare results in R

<img src="https://natur.gl/wp-content/uploads/2021/04/RensdyrTyre_CEgevang-2048x1365.jpg" alt="image info" />

In [ ]:
### make directory for the exercise and move into it
mkdir -p ~/kenya2026/relatedness
cd ~/kenya2026/relatedness

We will be using the data set of called genotypes from different reindeer populations in Greenland (cold!!).

Here is a map from Greenland with the different reindeer populations:

<img src="https://natur.gl/wp-content/uploads/2021/04/RensdyrBestande-1280x2106.png" alt="image info" />

## STEP 1 — Data Preparation

### 1a. Inspect the Reindeer dataset

In [ ]:
# How many individuals and SNPs?
wc -l /course/kenya2026/nuno/relatedness/Reindeer.fam
wc -l /course/kenya2026/nuno/relatedness/Reindeer.bim
  
# Population breakdown
cut -f2 /course/kenya2026/nuno/relatedness/Reindeer.fam | sort | uniq -c

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/kenya2026/exercises/Day4/quiz_dataset_summary.json')

The next command takes a while to run so go for a walk and run the next command in 5 minutes

In [ ]:
echo "This command was used to generate the precomputed files:"
echo "PCAone -b /course/kenya2026/nuno/relatedness/Reindeer -k 6 --ld -o pcaone"

echo
echo "[20/08/2026-10:46:11] initialize window-based RSVD (winSVD) with in-core 
[20/08/2026-10:46:11] permuting data matrix by columns in place 
[20/08/2026-10:46:23] running of epoch =  1, diff = 0.026404
[20/08/2026-10:46:28] running of epoch =  2, diff = 0.000399
[20/08/2026-10:46:32] running of epoch =  3, diff = 0.000230
[20/08/2026-10:46:37] running of epoch =  4, diff = 0.000031
PCAone winSVD converged but continues running to get S and V. 
[20/08/2026-10:46:42] running of epoch =  5, diff = 0.000083
PCAone winSVD converged but continues running to get S and V. 
[20/08/2026-10:46:46] running of epoch =  6, diff = 0.000084
[20/08/2026-10:46:46] stops at epoch =  7
[20/08/2026-10:46:46] ld-stats=0: calculate the ancestry adjusted LD matrix! 
[20/08/2026-10:46:50] save matched sites in .mbim file and permutation mode is  1
[20/08/2026-10:46:51] the LD matrix and SNPs info are saved 
[20/08/2026-10:46:51] eigen vectors and values saved 
[20/08/2026-10:46:51] PCAone - Randomized SVD done 
[20/08/2026-10:46:51] total elapsed reading time:  32.919000 seconds 
[20/08/2026-10:46:51] total elapsed wall time: 72.645000 seconds 
[20/08/2026-10:46:51] have a nice day. bye!"

echo
echo "Skipping this computationally intensive step during the course."

In [ ]:
  PCAone \
      -B /course/kenya2026/nuno/relatedness/pcaone.residuals \
      --match-bim /course/kenya2026/nuno/relatedness/pcaone.mbim \
      --ld-r2 0.1 \
      --ld-bp 1000000 \
      -o pcaone

In [ ]:
echo ----Apply pruning----
plink2 --bfile /course/kenya2026/nuno/relatedness/Reindeer \
  --extract pcaone.ld.prune.in \
  --maf 0.02 \
  --make-bed \
  --out Reindeer_pruned \
  --threads 4
echo ""
echo ---- Count SNPs----
echo "SNPs before pruning: $(wc -l < /course/kenya2026/nuno/relatedness/Reindeer.bim)"
echo "SNPs after pruning:  $(wc -l < Reindeer_pruned.bim)"

**Suggestion**: Always run LD pruning before KING/relatedness analysis, but **NOT** before FST or SFS estimation.
LD-pruned data biases allele frequency estimates.

**Which filter did we use?**

Use the plink site to understand better:
[Plink2 filters](https://www.cog-genomics.org/plink/2.0/filter)


In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/kenya2026/exercises/Day4/quiz_relatedness_preparation.json')

## STEP 2 — KING Kinship Estimation

PLINK2 implements the KING-robust estimator directly. It computes pairwise kinship for all sample pairs.

In [ ]:
echo ----Calculate kinship coefficients using KING----
plink2 --bfile Reindeer_pruned \
  --make-king-table \
  --out Reindeer_king \
  --threads 4

Output file: kinship_results.kin0 Columns: #IID1 IID2 NSNP HETHET IBS0 KINSHIP

In [ ]:
echo ---- Quick summary----
head Reindeer_king.kin0
echo ""
echo ---- Count pairs by degree----
awk 'NR>1 {
  if ($8 > 0.354)      print "Duplicate/MZ_twin"
  else if ($8 > 0.177) print "1st_degree"
  else if ($8 > 0.088) print "2nd_degree"
  else if ($8 > 0.044) print "3rd_degree"
  else                 print "Unrelated"
}' Reindeer_king.kin0 | sort | uniq -c | sort -rn

The values in the pair counts are huge ! **Can you think why?** 

**Are there any close relatives in this dataset? Which degree?**

In [ ]:
source("/course/kenya2026/nuno/relatedness/plot_pca_king.R")

plot_king_ibs0(
  king_file = "~/kenya2026/relatedness/Reindeer_king.kin0"
)

In [ ]:
cat("Device dimensions:", dev.size("px"), "\n")

source("/course/kenya2026/nuno/relatedness/plot_king_heatmap.R")

plot_king_heatmap(
  king_file = "~/kenya2026/relatedness/Reindeer_king.kin0"
)

**Does the first graph look familiar?** Remember the presentation

**And how do you interprete the red values in the heatmap?**


## STEP 3 — Remove the related individuals

Considering we have related individuals in the dataset, we should remove them. The stardart is to remove 1st degree or closer

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/kenya2026/exercises/Day4/quiz_king.json')

In [ ]:
# Extract pairs with kinship > 0.177 (1st degree or closer)
awk 'NR>1 && $8 > 0.177 {print $1"\n"$3}' Reindeer_king.kin0 \
  | sort -u > relatives_to_flag.txt

echo "Individuals to consider removing: $(wc -l < relatives_to_flag.txt)"

Now we should see if removing the related individuals has an impact in our results:

In [ ]:
echo ---- Calculate the pca with all individuals -----
plink2 \
  --bfile Reindeer_pruned \
  --pca 10 \
  --out pca_Wrelated
echo
echo ---- Calculate the pca after removing related individuals----
plink2 \
  --bfile Reindeer_pruned \
  --king-cutoff-table Reindeer_king.kin0 0.177 \
  --pca 10 \
  --out pca_Nonrelated

In [ ]:
options(
    repr.plot.width = 11,
    repr.plot.height = 8,
    repr.plot.res = 120
  )

source("/course/kenya2026/nuno/relatedness/plot_pca_king.R")

plot_pca_before_after(
    before_eigenvec = "~/kenya2026/relatedness/pca_Wrelated.eigenvec",
    after_eigenvec  = "~/kenya2026/relatedness/pca_Nonrelated.eigenvec",
    population_file = "/course/kenya2026/nuno/relatedness/population.tsv",
  )

**Do you see any differences? Is it important to remove related individuals?**

**Important**: In a real analysis you would remove one individual from each related pair (keeping the one with more data/higher coverage) before running ADMIXTURE, PCA, or FST.

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/kenya2026/exercises/Day4/quiz_relatedness_filtering.json')

# $F_{st}$ in wildebeest

We are back to wildebeest!

In this exercise we will cover:
 - Generating and displaying pairwise $F_{st}$ values
    
    
Tools used: plink2, R

The notebooks are editable, so feel free to experiment and change the code to see what happens or write notes in the text cells. Just remember to download the notebooks used here at some point if you want to save them with your own changes included.

In [ ]:
### make directory for the exercise
mkdir -p ~/kenya2026/Fst
cd ~/kenya2026/Fst

We will be using the data set of called genotypes from different blue wildebeest populations, as well as some black wildebeest as an outgroup to compare to, saved in a plink format file set. 

Here is the map from earlier to help show the sampling locations of the different wildebeest populations:
<img src="https://raw.githubusercontent.com/popgenDK/popgenDK.github.io/gh-pages/images/slider/wildeBeastMap.png" alt="image info" />


 **- Do you remember what a plink file set (.bed, bim and .fam) contains?**

In [ ]:
head /davidData/users/thomas/workshop/wildebeest_fst.fam

In [ ]:
head /davidData/users/thomas/workshop/wildebeest_fst.bim

Below we have the command used to run the $F_{st}$ estimation:

In [ ]:
plink2 --bfile /davidData/users/thomas/workshop/wildebeest_fst --within /davidData/users/thomas/workshop/clusterfile \
    --fst CATPHENO method=hudson --allow-extra-chr --threads 10

 **- How many individuals are in this files? And divided in how many populations?**

The cluster/within file give with `-within /davidData/users/thomas/workshop/clusterfile` tells the program how to separate the individuals into different groups for comparison. If we did not know up front which samples belonged together in populations, can you recall something we have looked at that could perhaps help with this?

Then let's have a look at the results:

In [ ]:
# some hartebeest samples were also originally included in this data set, but now we can just remove those from
# the output
grep -ve Hartebeest plink2.fst.summary > tmp
mv tmp plink2.fst.summary

# print the results
column -t plink2.fst.summary

 **- Which populations are most genetically differentiated? Which are most similar?**
 
 **- Can you indentify a pattern in the Fst values between black wildebeest and each of the blue wildebeest populations? Try to see if you can explain this pattern.**

Are each of these values large or small? This is quite difficult to answer without context, as it will depend on the type of data you are analyzing, the amount of data and the scope of your study. To provide context, one often looks at a matrix of $F_{st}$ values, which can be visualized using a heatmap. To do this we first need to transform the above data frame into a matrix, and then generate a heatmap using the heatmap.2-function.

In [ ]:
options(repr.matrix.max.cols=10, repr.matrix.max.rows=10)
options(repr.plot.width=16, repr.plot.height=16)
library(gplots)

# read the data into R
fst <- read.table("~/kenya2026/Fst/plink2.fst.summary")
names(fst) <- c("pop1", "pop2", "est")
fst <- fst[fst$pop1 != "Hartebeest" & fst$pop2 != "Hartebeest",]

Here we transform the table from above into a pairwise matrix that contains the exact same information, just in a different format:

In [ ]:
mat <- matrix(NA, 8, 8)
mat[lower.tri(mat)] <- fst$est
mat <- t(mat)
mat[lower.tri(mat)] <- fst$est
colnames(mat) <- c( "Amboseli", fst[1:7,2])
rownames(mat) <- c( "Amboseli", fst[1:7,2])
mat

In [ ]:
heatmap.2(mat, symm=T, trace='n', cexRow=1.5, cexCol=1.5, margins = c(12, 12))

**- Look at the clustering tree produced by this method. Do the different groups relate to each other as we would expect?**

**- We can see some discrete levels of values in the color key and in the histogram in the inset plot. What do these correspond to?**
 
An important note here is that the tree/dendrogram used to order the groups here simply comes from clustering based on the $F_{st}$ values and will not neccesarily reflect the true evolutionary history of the groups.

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/kenya2026/exercises/Day4/quiz_wildebeest_fst.json')

# Extra/optional - Reindeer SAF-based $F_{ST}$

We now estimate genotype-likelihood-based differentiation for **Qassit**, **Neria**, and **Ameralik**. The ANGSD SAF files are supplied in `/course/kenya2026/nuno/relatedness`; do not regenerate them.

Each population needs a matching `.saf.idx`, `.saf.gz`, and `.saf.pos.gz` triplet. These SAFs must be full-dimensional (not generated with `ANGSD -fold 1`) and must use the same reference and compatible filters.

`winsfs` estimates the pairwise 2D-SFS. We then remove its `#SHAPE` header and use ANGSD's `realSFS fst` functions for global and windowed $F_{ST}$.

In [ ]:
mkdir -p ~/kenya2026/reindeer_saf_fst
cd ~/kenya2026/reindeer_saf_fst

SAF_DIR=/course/kenya2026/nuno/relatedness
pops=(Qassit Neria Ameralik)
extensions=(saf.idx saf.gz saf.pos.gz)
missing=0

for pop in "${pops[@]}"; do
  for extension in "${extensions[@]}"; do
    file="$SAF_DIR/$pop.$extension"
    if [[ ! -s "$file" ]]; then
      echo "MISSING: $file" >&2
      missing=1
    fi
  done
done

if (( missing )); then
  echo "Add all nine supplied SAF components to $SAF_DIR before continuing." >&2
  exit 1
fi

echo "All supplied SAF components were found."
ls -lh "$SAF_DIR"/{Qassit,Neria,Ameralik}.saf.{idx,gz,pos.gz}

In [ ]:
echo "This command was used to generate the precomputed files:"
echo "SAF_DIR=/course/kenya2026/nuno/relatedness
OUT=~/kenya2026/reindeer_saf_fst
THREADS=40

pairs=("Qassit Neria" "Qassit Ameralik" "Neria Ameralik")

for pair in "${pairs[@]}"; do
  read -r pop1 pop2 <<< "$pair"
  prefix="$OUT/${pop1}_${pop2}"
  echo "Estimating $pop1 versus $pop2"

  winsfs --threads "$THREADS" --seed 2026 \
    "$SAF_DIR/$pop1.saf.idx" "$SAF_DIR/$pop2.saf.idx" \
    > "$prefix.winsfs"

  # winsfs writes a #SHAPE header; realSFS fst expects only numeric SFS values.
  tail -n 1 "$prefix.winsfs" > "$prefix.2dsfs"

  realSFS fst index \
    "$SAF_DIR/$pop1.saf.idx" "$SAF_DIR/$pop2.saf.idx" \
    -sfs "$prefix.2dsfs" -fstout "$prefix"

  realSFS fst stats "$prefix.fst.idx" > "$prefix.global_fst.txt"
  realSFS fst stats2 "$prefix.fst.idx" -win 100000 -step 50000 \
    > "$prefix.100kb_fst.tsv"
done"

echo
echo "Estimating Qassit versus Neria
Estimating Qassit versus Ameralik
Estimating Neria versus Ameralik"

In [ ]:
echo ----Global pairwise Fst results----
for result in /course/kenya2026/nuno/relatedness/*.global_fst.txt; do
  echo "Population Pair  100kb    global"
  printf '%s: ' "$(basename "$result" .global_fst.txt)"
  cat "$result"
  echo
done

**Which population pair has the largest global $F_{ST}$? Which has the smallest?**

**Do particular 100-kb windows show much stronger differentiation than the global estimate?**

**Why must the three SAF datasets use compatible genomic sites, references, and filters?**

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/kenya2026/exercises/Day4/quiz_reindeer_saf_fst.json')

# Reindeer admixture-aware relatedness with relateAdmix

`relateAdmix` estimates relatedness while conditioning on individual ancestry. It combines the LD-pruned PLINK dataset with ADMIXTURE's individual ancestry proportions (`.Q`) and ancestral-population allele frequencies (`.P`).

We reuse `Reindeer_pruned` from the PCAone/PLINK section. The `.Q` rows must match FAM sample order, and the `.P` rows must match BIM marker order.

In [ ]:
cd ~/kenya2026/relatedness

# K=2 keeps this classroom run short. A full analysis should compare K values and seeds.
admixture --cv=5 -s 2026 Reindeer_pruned.bed 2 | tee admixture.K2.log
echo
echo ----Check matching dimensions----
wc -l Reindeer_pruned.fam Reindeer_pruned.2.Q
wc -l Reindeer_pruned.bim Reindeer_pruned.2.P

In [ ]:
## If it takes too long, remove the hastag from this next command and run it.
#cp /course/kenya2026/nuno/relatedness/Reindeer_pruned.2.* ~/kenya2026/relatedness/.

### Plot the ADMIXTURE results

Each vertical bar represents one individual, and each colour represents the proportion assigned to one of the two inferred ancestry components. Individuals are grouped by population.

In [ ]:
work_dir <- path.expand("~/kenya2026/relatedness")

Q <- as.matrix(read.table(file.path(work_dir, "Reindeer_pruned.2.Q")))
fam <- read.table(file.path(work_dir, "Reindeer_pruned.fam"),
                  stringsAsFactors = FALSE)
stopifnot(nrow(Q) == nrow(fam))

population <- fam[, 2]
order_by_population <- order(population)
Q <- Q[order_by_population, , drop = FALSE]
population <- population[order_by_population]

ancestry_colours <- hcl.colors(ncol(Q), palette = "Dark 3")
population_runs <- rle(population)
group_ends <- cumsum(population_runs$lengths)
group_starts <- c(1, head(group_ends, -1) + 1)

options(repr.plot.width = 14, repr.plot.height = 6)
par(mar = c(9, 4, 3, 1))
bar_positions <- barplot(
  t(Q), col = ancestry_colours, border = NA, space = 0, axes = FALSE,
  ylab = "Ancestry proportion", main = "Reindeer ADMIXTURE results (K = 2)"
)
axis(2, las = 1)
group_centres <- vapply(seq_along(group_starts), function(i)
  mean(bar_positions[group_starts[i]:group_ends[i]]), numeric(1))
axis(1, at = group_centres, labels = population_runs$values,
     las = 2, tick = FALSE, cex.axis = 0.8)
if (length(group_ends) > 1)
  abline(v = bar_positions[group_ends[-length(group_ends)]] + 0.5,
         col = "white", lwd = 1)
legend("topright", legend = paste0("Ancestry ", seq_len(ncol(Q))),
       fill = ancestry_colours, border = NA, bty = "n")

**Which are the major problems with the previous step, considering what you learned about admixture results validation yesterday?**

### Calculate relateAdmix

In [ ]:
cd ~/kenya2026/relatedness
RELATEADMIX=/course/kenya2026/nuno/relatedness/relateAdmix

"$RELATEADMIX" -plink Reindeer_pruned \
  -fname Reindeer_pruned.2.P \
  -qname Reindeer_pruned.2.Q \
  -o Reindeer_relateAdmix_K2 -P 4

echo ----First relateAdmix output rows----
head Reindeer_relateAdmix_K2

**Do the `.Q` and FAM row counts match? Do the `.P` and BIM row counts match?** You should check the results for FAM and BIM that you measured before. Please run the command yourself - open a new promp and make sure you are using bash (to the right) and run the command that you used above to measure it in the FAM and BIM

### Comparing KING and relateAdmix relatedness estimates

KING estimates kinship directly from the genotype data, whereas relateAdmix accounts for each individual's estimated ancestry.

  RelateAdmix reports the probabilities that a pair shares zero, one, or two alleles identical by descent (`k0`, `k1`, and `k2`).
  We convert these probabilities to a kinship coefficient:

  \[
  \phi = \frac{k_1}{4} + \frac{k_2}{2}
  \]

  The scatter plot compares this ancestry-adjusted estimate with the KING estimate. The dashed lines indicate the first-degree
  kinship threshold of 0.177.

  Pairs in the upper-right quadrant are classified as first-degree relatives by both methods. Pairs to the right of the vertical
  line but below the horizontal line have high KING kinship but lower ancestry-adjusted relatedness.

In [ ]:
##read and combine the results

work_dir <- path.expand("~/kenya2026/relatedness")

# Read the PLINK sample information.
fam <- read.table(
  file.path(work_dir, "Reindeer_pruned.fam"),
  stringsAsFactors = FALSE
 )
colnames(fam)[1:2] <- c("FID", "IID")

# Read relateAdmix output.
relate <- read.table(
  file.path(work_dir, "Reindeer_relateAdmix_K2"),
  header = TRUE,
  stringsAsFactors = FALSE
  )

# relateAdmix uses zero-based row numbers from the FAM file.
relate$FID1 <- fam$FID[relate$ind1 + 1]
relate$IID1 <- fam$IID[relate$ind1 + 1]
relate$FID2 <- fam$FID[relate$ind2 + 1]
relate$IID2 <- fam$IID[relate$ind2 + 1]

# Convert IBD probabilities to a kinship coefficient.
relate$RELATE_KINSHIP <- relate$k1 / 4 + relate$k2 / 2

# Read the KING table.
king <- read.table(
  file.path(work_dir, "Reindeer_king.kin0"),
  header = TRUE,
  comment.char = "",
  check.names = FALSE,
  stringsAsFactors = FALSE
  )

# PLINK may place "#" before the first column name.
names(king) <- sub("^#", "", names(king))

# Create an order-independent identifier for each pair.
pair_key <- function(fid1, iid1, fid2, iid2) {
  sample1 <- paste(fid1, iid1, sep = ":")
  sample2 <- paste(fid2, iid2, sep = ":")

  ifelse(
    sample1 < sample2,
    paste(sample1, sample2, sep = " | "),
    paste(sample2, sample1, sep = " | ")
  )
}

relate$PAIR <- pair_key(
  relate$FID1, relate$IID1,
  relate$FID2, relate$IID2
)

king$PAIR <- pair_key(
  king$FID1, king$IID1,
  king$FID2, king$IID2
)

comparison <- merge(
  king[, c("PAIR", "KINSHIP")],
  relate[, c(
    "PAIR", "FID1", "IID1", "FID2", "IID2",
    "k0", "k1", "k2", "RELATE_KINSHIP"
  )],
  by = "PAIR"
)

cat("KING pairs:", nrow(king), "\n")
cat("relateAdmix pairs:", nrow(relate), "\n")
cat("Matched pairs:", nrow(comparison), "\n")

### KING versus relateAdmix

  Each point represents a pair of individuals:

  - Grey: below the threshold with both methods.
  - Purple: above the threshold with both methods.
  - Orange: above the threshold with KING only.
  - Blue: above the threshold with relateAdmix only.

  The diagonal line represents equal estimates from the two methods.

In [ ]:
##main scatter plot

  kinship_cutoff <- 0.177

  comparison$classification <- with(
    comparison,
    ifelse(
      KINSHIP >= kinship_cutoff &
        RELATE_KINSHIP >= kinship_cutoff,
      "Both methods",
      ifelse(
        KINSHIP >= kinship_cutoff,
        "KING only",
        ifelse(
          RELATE_KINSHIP >= kinship_cutoff,
          "relateAdmix only",
          "Below threshold"
        )
      )
    )
  )

  point_colours <- c(
    "Below threshold" = adjustcolor("grey50", alpha.f = 0.35),
    "Both methods" = "#7B3294",
    "KING only" = "#E66101",
    "relateAdmix only" = "#0571B0"
  )

  options(repr.plot.width = 8, repr.plot.height = 7)

  plot(
    comparison$KINSHIP,
    comparison$RELATE_KINSHIP,
    pch = 16,
    cex = 0.8,
    col = point_colours[comparison$classification],
    xlab = "KING kinship coefficient",
    ylab = "relateAdmix kinship coefficient",
    main = "KING versus ancestry-adjusted relatedness"
  )

  # Equal estimates
  abline(a = 0, b = 1, col = "grey50", lty = 3)

  # First-degree threshold
  abline(
    v = kinship_cutoff,
    h = kinship_cutoff,
    col = "firebrick",
    lty = 2,
    lwd = 2
  )

  legend(
    "topleft",
    legend = names(point_colours),
    col = point_colours,
    pch = 16,
    bty = "n"
  )

In [ ]:
### Optional: inspect discordant pairs

  discordant_pairs <- subset(
    comparison,
    (KINSHIP >= kinship_cutoff) !=
      (RELATE_KINSHIP >= kinship_cutoff)
  )

  discordant_pairs <- discordant_pairs[
    order(
      abs(
        discordant_pairs$KINSHIP -
          discordant_pairs$RELATE_KINSHIP
      ),
      decreasing = TRUE
    ),
  ]

  discordant_pairs[, c(
    "FID1", "IID1", "FID2", "IID2",
    "KINSHIP", "RELATE_KINSHIP", "classification"
  )]

### Heatmap of ancestry-adjusted relatedness

  The heatmap summarizes relateAdmix kinship estimates for every pair of individuals. Samples are ordered by population so that
  within-population and between-population patterns are easier to recognize.

  The diagonal is left blank because it represents self-comparisons rather than pairwise relatedness estimates.

In [ ]:
###supporting heatmap

  sample_labels <- paste(fam$FID, fam$IID, sep = ":")
  number_of_samples <- nrow(fam)

  relate_matrix <- matrix(
    0,
    nrow = number_of_samples,
    ncol = number_of_samples,
    dimnames = list(sample_labels, sample_labels)
  )

  row_index <- relate$ind1 + 1
  column_index <- relate$ind2 + 1

  relate_matrix[
    cbind(row_index, column_index)
  ] <- relate$RELATE_KINSHIP

  relate_matrix[
    cbind(column_index, row_index)
  ] <- relate$RELATE_KINSHIP

  diag(relate_matrix) <- NA

  # The second FAM column contains the population labels in this dataset.
  population <- fam$IID
  sample_order <- order(population)

  relate_matrix <- relate_matrix[
    sample_order,
    sample_order
  ]

  ordered_population <- population[sample_order]
  population_runs <- rle(ordered_population)
  group_ends <- cumsum(population_runs$lengths)
  group_starts <- c(1, head(group_ends, -1) + 1)
  group_centres <- (group_starts + group_ends) / 2

  heat_colours <- colorRampPalette(
    c("white", "#FEE8C8", "#FDBB84", "#E34A33", "#7F0000")
  )(100)

  options(repr.plot.width = 10, repr.plot.height = 9)
  par(mar = c(9, 9, 3, 7))

  kinship_range <- range(relate_matrix, na.rm = TRUE)

  image(
    x = seq_len(number_of_samples),
    y = seq_len(number_of_samples),
    z = t(relate_matrix[number_of_samples:1, ]),
    col = heat_colours,
    zlim = kinship_range,
    axes = FALSE,
    xlab = "",
    ylab = "",
    main = "relateAdmix ancestry-adjusted kinship"
  )

  axis(
    1,
    at = group_centres,
    labels = population_runs$values,
    las = 2,
    tick = FALSE,
    cex.axis = 0.7
  )

  axis(
    2,
    at = number_of_samples + 1 - rev(group_centres),
    labels = rev(population_runs$values),
    las = 2,
    tick = FALSE,
    cex.axis = 0.7
  )

  if (length(group_ends) > 1) {
    boundaries <- group_ends[-length(group_ends)] + 0.5

    abline(v = boundaries, col = "grey70", lwd = 0.7)
    abline(
      h = number_of_samples + 0.5 - boundaries,
      col = "grey70",
      lwd = 0.7
    )
  }

  # Add the colour legend after drawing the heatmap.
  legend_values <- pretty(kinship_range, n = 5)
  legend_values <- legend_values[
    legend_values >= kinship_range[1] &
      legend_values <= kinship_range[2]
  ]

  if (diff(kinship_range) == 0) {
    legend_colours <- heat_colours[50]
  } else {
    legend_colours <- heat_colours[
      round(
        1 + 99 *
          (legend_values - kinship_range[1]) /
          diff(kinship_range)
      )
    ]
  }

  legend(
    "right",
    inset = c(-0.19, 0),
    legend = format(legend_values, digits = 3, trim = TRUE),
    fill = legend_colours,
    border = NA,
    title = "Kinship",
    bty = "n",
    xpd = NA,
    cex = 0.75
  )

**Compare close pairs from KING and relateAdmix. Could population structure explain any disagreement?**

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/kenya2026/exercises/Day4/quiz_relateadmix.json')